# A1/A2 — Exploratory Data Analysis and Structure Detection

## Bengali crop-disease corpus

This local-first notebook examines the scanned page images used by the knowledge-base pipeline. It produces evidence for the A1/A2 forms and informs the implementations of preprocessing, layout detection, Bangla OCR, chunking, and indexing.

It deliberately separates three things:

- **Exploratory diagnostics:** image quality and lightweight structure measurements used to understand the corpus.
- **Evaluation data:** `grading_kit/heldout_pages/` and `labels.jsonl`; these are never used to tune or train OCR.
- **Production choices:** changes made only after the findings are reproduced in the pipeline and documented in `configs/design_choices.md`.

> **Kaggle note:** This notebook remains local by default. If running in Kaggle, change only the path configuration cell to point at your mounted dataset under `/kaggle/input/...`; do not copy data into a new project layout or change the analysis logic.

## Questions this notebook answers

1. Does the corpus meet the image/page-count requirements, and are files readable?
2. What scan conditions occur: resolution, skew, blur, contrast, border/shadow contamination, and ink coverage?
3. What page structures occur: dense text, headings, lists, sparse pages, likely tables/figures, and possible multi-column layouts?
4. What Bangla and mixed-script characteristics appear in the **labelled held-out evaluation text**?
5. Which pages represent normal, difficult, and structurally complex cases for visual inspection and future OCR evaluation?
6. Is the corpus split safely by document, with held-out pages excluded from the build corpus?

The notebook does not claim OCR quality: that belongs in `kb_demo.ipynb` after an OCR model has been run against held-out transcriptions.

## 0. Local setup

Install local analysis dependencies once if they are not already available:

```bash
pip install pillow opencv-python pandas matplotlib pyyaml
```

> **Kaggle note:** Prefer Kaggle's preinstalled packages. If an install is necessary, use a separate setup cell such as `!pip install -q <package>`. Some OCR-model downloads require Kaggle Internet to be enabled or model files attached as a Kaggle Dataset.

In [ ]:
from __future__ import annotations

import json
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image, UnidentifiedImageError

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110

IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
BENGALI_PATTERN = re.compile(r"[\u0980-\u09FF]")
LATIN_PATTERN = re.compile(r"[A-Za-z]")


## 1. Paths and project configuration

The default paths are relative to `notebooks/`, so this notebook should be launched from this repository. The raw corpus must use the same `data/raw/` location read by `ingest/loader.py`.

> **Kaggle note:** Set `PROJECT_ROOT` to the mounted dataset directory, for example `Path('/kaggle/input/g16-bangla-crop-disease/doc-agent-starter')`. Keep `RUNNING_ON_KAGGLE = False` locally; setting it to `True` changes paths only.

In [ ]:
RUNNING_ON_KAGGLE = False

# Local default. Change this one line only when using a Kaggle mounted dataset.
PROJECT_ROOT = Path.cwd().parent
if RUNNING_ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/input/<your-dataset-slug>/doc-agent-starter")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
HELDOUT_DIR = PROJECT_ROOT / "grading_kit" / "heldout_pages"
LABELS_PATH = PROJECT_ROOT / "grading_kit" / "labels.jsonl"
TASK_PATH = PROJECT_ROOT / "configs" / "task.yaml"

assert PROJECT_ROOT.is_dir(), f"Project root not found: {PROJECT_ROOT}"
assert TASK_PATH.is_file(), f"Missing task configuration: {TASK_PATH}"

with TASK_PATH.open(encoding="utf-8") as source:
    task = yaml.safe_load(source)

print(f"Project: {task['group_id']} | {task['domain']}")
print(f"Speciality: {task['data_speciality']} | NFR: {task['primary_nfr']}")
print(f"Raw corpus: {RAW_DIR}")


## 2. Corpus inventory and file-integrity check

This mirrors the loader's image-extension policy. A document ID is the parent directory relative to `data/raw/`; therefore, put each separately splittable source document in its own folder. A flat `data/raw/` directory is treated as a single document named `default`.

This cell checks files without mutating them. It records unreadable images rather than hiding them.

In [ ]:
def discover_pages(raw_dir: Path) -> pd.DataFrame:
    if not raw_dir.is_dir():
        raise FileNotFoundError(f"Raw corpus directory does not exist: {raw_dir}")

    records: list[dict[str, object]] = []
    for path in sorted(raw_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        relative = path.relative_to(raw_dir)
        document = relative.parent.as_posix() if relative.parent != Path(".") else "default"
        page_id = relative.with_suffix("").as_posix()
        record: dict[str, object] = {
            "page_id": page_id,
            "doc_id": document,
            "relative_path": relative.as_posix(),
            "path": path,
            "extension": path.suffix.lower(),
            "size_kib": round(path.stat().st_size / 1024, 2),
            "readable": True,
            "width": np.nan,
            "height": np.nan,
            "mode": None,
        }
        try:
            with Image.open(path) as image:
                record.update(width=image.width, height=image.height, mode=image.mode)
        except (OSError, UnidentifiedImageError):
            record["readable"] = False
        records.append(record)

    if not records:
        raise ValueError(f"No supported page images found in {raw_dir}")
    return pd.DataFrame(records)

pages_df = discover_pages(RAW_DIR)
min_pages = int(task["corpus"]["min_pages"])

print(f"Pages found: {len(pages_df):,} (minimum: {min_pages:,})")
print(f"Documents found: {pages_df['doc_id'].nunique():,}")
print(f"Readable pages: {pages_df['readable'].sum():,}/{len(pages_df):,}")
display(pages_df.groupby('doc_id').agg(pages=('page_id', 'count'), size_mib=('size_kib', lambda x: round(x.sum() / 1024, 2)), readable=('readable', 'all')))

if len(pages_df) < min_pages:
    print("WARNING: page-count requirement is not yet met.")
if not pages_df['readable'].all():
    display(pages_df.loc[~pages_df['readable'], ['page_id', 'relative_path']])


## 3. Image-quality diagnostics

These are interpretable diagnostics, not labels of OCR quality. They identify pages to inspect and support preprocessing decisions:

- Laplacian variance: lower values often indicate blur.
- grayscale standard deviation: low values often indicate weak contrast.
- Otsu foreground ratio: unusually low/high values identify sparse, blank, or dark pages.
- border-ink ratio: a possible gutter shadow, border artifact, or crop issue.
- Hough-line median angle: an approximate skew estimate.
- connected-component statistics: a coarse view of broken strokes/noise.

Thresholds must be selected only after viewing examples; do not treat one heuristic score as ground truth.

In [ ]:
def estimate_skew(binary: np.ndarray) -> float:
    edges = cv2.Canny(binary, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, 80, minLineLength=80, maxLineGap=12)
    if lines is None:
        return 0.0
    angles = [np.degrees(np.arctan2(y2 - y1, x2 - x1)) for x1, y1, x2, y2 in lines[:, 0]]
    near_horizontal = [angle for angle in angles if -15 <= angle <= 15]
    return float(np.median(near_horizontal)) if near_horizontal else 0.0

def analyse_image(path: Path) -> dict[str, float]:
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"OpenCV could not read {path}")
    height, width = gray.shape
    _threshold, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    foreground = binary > 0
    margin = max(1, int(0.05 * min(height, width)))
    border = np.ones_like(foreground, dtype=bool)
    border[margin:height - margin, margin:width - margin] = False
    labels, _labels, stats, _centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
    areas = stats[1:labels, cv2.CC_STAT_AREA] if labels > 1 else np.array([])

    return {
        "width": float(width),
        "height": float(height),
        "aspect_ratio": float(height / width),
        "mean_brightness": float(gray.mean()),
        "contrast_std": float(gray.std()),
        "blur_laplacian_var": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        "foreground_ratio": float(foreground.mean()),
        "border_ink_ratio": float(foreground[border].mean()),
        "skew_degrees": estimate_skew(binary),
        "component_count": float(len(areas)),
        "median_component_area": float(np.median(areas)) if len(areas) else 0.0,
    }

quality_rows = []
for row in pages_df.loc[pages_df.readable].itertuples(index=False):
    quality_rows.append({"page_id": row.page_id, **analyse_image(row.path)})

# width and height already come from the file-inventory pass. Keep those canonical
# columns and suffix the repeated image-analysis measurements to avoid width_x/width_y.
quality_df = pages_df.merge(
    pd.DataFrame(quality_rows), on="page_id", how="left", suffixes=("", "_metric")
)
display(quality_df.groupby('doc_id')[['width', 'height', 'contrast_std', 'blur_laplacian_var', 'foreground_ratio', 'skew_degrees']].agg(['median', 'min', 'max']).round(2))


In [ ]:
metrics = ['contrast_std', 'blur_laplacian_var', 'foreground_ratio', 'border_ink_ratio', 'skew_degrees', 'component_count']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, metric in zip(axes.flat, metrics):
    for document, subset in quality_df.groupby('doc_id'):
        axis.hist(subset[metric].dropna(), bins=30, alpha=0.5, label=document)
    axis.set_title(metric)
    axis.set_xlabel(metric)
axes[0, 0].set_ylabel('page count')
if quality_df['doc_id'].nunique() <= 8:
    axes[0, 0].legend(fontsize=8)
plt.suptitle('Scan-quality distributions by source document', y=1.02)
plt.tight_layout()
plt.show()

# Review candidates: these are diagnostic samples, not automatically discarded pages.
review_columns = ['page_id', 'doc_id', 'contrast_std', 'blur_laplacian_var', 'foreground_ratio', 'border_ink_ratio', 'skew_degrees']
print('Lowest-contrast pages:')
display(quality_df.nsmallest(8, 'contrast_std')[review_columns])
print('Most-skewed pages:')
display(quality_df.assign(abs_skew=quality_df.skew_degrees.abs()).nlargest(8, 'abs_skew')[review_columns])


## 4. Lightweight page-structure diagnostics

This is not the final Stage 2 layout detector. It is a transparent EDA baseline that estimates structure from projection profiles and connected components. It highlights pages that require careful reading order, table/figure handling, or manual inspection.

A production layout stage must return the locked `Region(page_id, bbox, kind)` contract; these diagnostics help choose and test that stage.

In [ ]:
def structure_diagnostics(path: Path) -> dict[str, float]:
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    _threshold, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    ink = (binary > 0).astype(np.uint8)
    horizontal = ink.mean(axis=1)
    vertical = ink.mean(axis=0)

    # Long low-ink runs in the vertical profile are candidates for column gutters.
    low_vertical = vertical < max(0.01, np.quantile(vertical, 0.15))
    transitions = np.diff(np.r_[False, low_vertical, False].astype(int))
    starts, ends = np.where(transitions == 1)[0], np.where(transitions == -1)[0]
    gutter_widths = ends - starts
    width = ink.shape[1]
    possible_gutters = int(np.sum(gutter_widths >= max(8, width * 0.015)))

    # Dense horizontal peaks approximate text-line activity; isolated large components can flag figures/tables.
    active_rows = horizontal > max(0.01, np.quantile(horizontal, 0.55))
    row_changes = np.diff(np.r_[False, active_rows, False].astype(int))
    line_bands = int(np.sum(row_changes == 1))
    components, _labels, stats, _centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
    areas = stats[1:components, cv2.CC_STAT_AREA] if components > 1 else np.array([])
    large_regions = int(np.sum(areas > 0.01 * ink.size))

    return {
        'estimated_line_bands': float(line_bands),
        'possible_column_gutters': float(possible_gutters),
        'large_nontext_region_candidates': float(large_regions),
    }

structure_rows = []
for row in quality_df.loc[quality_df.readable].itertuples(index=False):
    structure_rows.append({'page_id': row.page_id, **structure_diagnostics(row.path)})

structure_df = quality_df.merge(pd.DataFrame(structure_rows), on='page_id', how='left')
display(structure_df.groupby('doc_id')[['estimated_line_bands', 'possible_column_gutters', 'large_nontext_region_candidates']].agg(['median', 'max']).round(1))

complex_pages = structure_df.sort_values(['possible_column_gutters', 'large_nontext_region_candidates', 'estimated_line_bands'], ascending=False)
display(complex_pages[['page_id', 'doc_id', 'possible_column_gutters', 'large_nontext_region_candidates', 'estimated_line_bands']].head(12))


In [ ]:
def show_structure(page_row: pd.Series) -> None:
    gray = cv2.imread(str(page_row.path), cv2.IMREAD_GRAYSCALE)
    _threshold, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    ink = binary > 0
    fig, axes = plt.subplots(1, 3, figsize=(16, 6), gridspec_kw={'width_ratios': [4, 1, 1]})
    axes[0].imshow(gray, cmap='gray')
    axes[0].set_title(f"{page_row.page_id} — visual inspection")
    axes[0].axis('off')
    axes[1].plot(ink.mean(axis=1), np.arange(ink.shape[0]))
    axes[1].invert_yaxis()
    axes[1].set_title('row ink')
    axes[2].plot(np.arange(ink.shape[1]), ink.mean(axis=0))
    axes[2].set_title('column ink')
    plt.tight_layout()
    plt.show()

# Inspect a normal, a low-quality, and a structurally complex page. Replace IDs after reviewing output.
normal_page = structure_df.iloc[len(structure_df) // 2]
low_quality_page = structure_df.nsmallest(1, 'contrast_std').iloc[0]
complex_page = complex_pages.iloc[0]
for candidate in (normal_page, low_quality_page, complex_page):
    show_structure(candidate)


## 5. Held-out labels: Bangla and mixed-script analysis

The labels are evaluation-only. This section examines their script composition to justify Bangla-aware OCR and multilingual embeddings, but it must not be used to select OCR hyperparameters. It also verifies that label page IDs correspond to held-out images.

If labels are not ready yet, this section reports that fact and the rest of the notebook remains usable.

In [ ]:
def load_labels(path: Path) -> pd.DataFrame:
    if not path.is_file():
        return pd.DataFrame(columns=['page_id', 'text'])
    rows = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
        if line.strip():
            record = json.loads(line)
            if not {'page_id', 'text'} <= record.keys():
                raise ValueError(f"labels.jsonl line {line_number} needs page_id and text")
            rows.append({'page_id': str(record['page_id']), 'text': str(record['text'])})
    return pd.DataFrame(rows)

labels_df = load_labels(LABELS_PATH)
heldout_images = {path.stem for path in HELDOUT_DIR.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS} if HELDOUT_DIR.is_dir() else set()

if labels_df.empty:
    print('No usable held-out labels found yet. Add real page images and exact human transcriptions before A2 evidence collection.')
else:
    def script_summary(text: str) -> pd.Series:
        normalized = unicodedata.normalize('NFC', text)
        nonspace = [character for character in normalized if not character.isspace()]
        return pd.Series({
            'characters': len(normalized),
            'words': len(normalized.split()),
            'bangla_characters': sum(bool(BENGALI_PATTERN.fullmatch(character)) for character in nonspace),
            'latin_characters': sum(bool(LATIN_PATTERN.fullmatch(character)) for character in nonspace),
            'contains_latin': bool(LATIN_PATTERN.search(normalized)),
        })

    label_stats = labels_df.join(labels_df.text.apply(script_summary))
    display(label_stats[['page_id', 'words', 'bangla_characters', 'latin_characters', 'contains_latin']])
    print(f"Held-out labels: {len(label_stats)} | labelled words: {label_stats.words.sum():,}")
    print(f"Labels containing Latin terms: {label_stats.contains_latin.mean():.1%}")
    missing_images = sorted(set(label_stats.page_id) - heldout_images)
    if missing_images:
        print('WARNING: label IDs without matching held-out image stems:', missing_images)


## 6. Split and leakage audit

The assignment requires a split by whole source document, never by page or chapter. This notebook checks the prerequisite: more than one `doc_id` must exist to support document-level train/validation/test separation.

A single book can still support held-out OCR pages for measurement, but it cannot honestly claim a document-level model split. Add independent source documents if you need train/validation/test splits for tuning.

In [ ]:
document_counts = pages_df.groupby('doc_id').size().sort_values(ascending=False)
display(document_counts.to_frame('pages'))

if len(document_counts) < 2:
    print('WARNING: only one source document is present. Do not describe a chapter/page split as a document-level split.')
else:
    # Record the final document-to-split assignment in provenance.md/config, then replace this example map.
    split_by_document = {document: 'UNASSIGNED' for document in document_counts.index}
    split_df = pages_df.assign(split=pages_df.doc_id.map(split_by_document))
    assert split_df.groupby('doc_id').split.nunique().eq(1).all(), 'A document appears in multiple splits.'
    display(split_df.groupby('split').agg(documents=('doc_id', 'nunique'), pages=('page_id', 'count')))

# Held-out pages must not be placed under data/raw/, so the loader cannot ingest them.
raw_stems = {Path(path).stem for path in pages_df.path}
overlap = raw_stems & heldout_images
print(f"Raw/held-out filename-stem overlap: {len(overlap)}")
if overlap:
    print('Inspect these names: same stems are not proof of leakage, but duplicate image files must not enter both locations.', sorted(overlap)[:10])


## 7. Findings to carry into A2

After running the notebook, replace this checklist with measured findings—not generic statements. It should feed `data/provenance.md`, `configs/design_choices.md`, `reports/pipeline_diagram.md`, and Sections 2–4 of the A2 form.

- **Corpus scope:** `___` images across `___` documents; `___` readable; `___` MiB.
- **Scan quality:** report medians/ranges and name representative difficult page IDs.
- **Structure:** state whether headings, lists, tables/figures, or possible columns occur, based on visual inspection of flagged pages.
- **Bangla characteristics:** report script/mixed-Latin evidence from held-out labels.
- **Pipeline implication:** name the preprocessing, layout, OCR, and embedding decisions each finding motivates.
- **Limitations:** describe what the heuristics cannot determine and what the A2 OCR/layout evaluations must test.

For the auditable NFR, preserve the notebook version, source corpus SHA-256 snapshot, configuration, selected page IDs, and measured outputs with the index build metadata.